# 📝 가설검정·회귀 과제 LV1(기초) — 가설검정 기초

> 이 단원에서 배운 **가설검정**(정규성 검정, t-검정, 비모수 검정, 분산분석, 카이제곱)을 타이타닉 승객 데이터로 **한 문제에 하나씩** 확인하는 과제입니다. 각 검정의 **검정통계량**과 **p-value**(그리고 필요하면 **효과크기**)를 직접 계산해 봅니다.

## 풀이 방법
1. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
2. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
3. 막히면 `힌트` 를 펼쳐 보세요.

> **자가채점이 없는 부분**: 그래프 문제(11)와 서술형(1·13·14), 그리고 2·7·12번의 **판단·해석 서술** 부분입니다(12번은 정량 자가채점 + 효과크기 해석 서술이 함께 있어요). 서술 부분은 정답 노트북의 모범 서술·완성 그래프와 비교하세요.

- 데이터는 `data/titanic.csv`(타이타닉 승객) 를 씁니다. 주요 열: `survived`(생존 0/1), `pclass`(객실 등급 1/2/3), `sex`(male/female), `age`(나이), `fare`(요금).
- **`age` 열에는 결측치가 177개** 있습니다. 나이로 검정할 때는 반드시 `dropna()` 로 결측을 먼저 제거하세요.
- **문제 5(대응표본 t-검정)만 예외로** 별도의 작은 데이터(`study_scores.csv`, 학습 프로그램 참가자 15명의 전/후 점수)를 씁니다. 그 문제 안에서 다시 안내합니다.
- 문제마다 데이터를 새로 불러오거나(`pd.read_csv`) 필요한 열만 골라 `dropna()` 하면 앞 문제의 변형에 영향받지 않아요.

화이팅!

아래 셀을 먼저 실행해 통계 검정 라이브러리와 한글 폰트를 준비하세요.

In [ ]:
# [제공 코드] 통계 검정에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')   # pingouin 의 사소한 경고를 숨겨 출력을 깔끔하게
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg              # ★ 이번 과제 주력: 검정+효과크기+신뢰구간을 한 표로
from scipy import stats            # 카이제곱 적합도(문제 10)에만 사용 — pingouin 미지원

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('data/titanic.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[수치 요약] describe()"); display(df.describe())
print("\n[범주형 요약] describe(exclude='number')"); display(df.describe(exclude='number'))
print("\n[age 결측 개수]:", df["age"].isna().sum())

## 1. 데이터 살펴보기 (서술형)
**배경**: 검정에 들어가기 전에 데이터를 **눈으로 파악**하는 것이 첫걸음입니다. 위 `데이터 살펴보기` 셀의 `head()`·`info()`·`describe()` 출력을 보고, 이 타이타닉 데이터에 대해 알게 된 사실을 정리해 보세요.

**요구사항**:
- 아래 서술 셀에 이 데이터에 대한 **관찰 2~3가지**를 문장으로 적으세요.
- 예를 들어: 생존자와 사망자 수, 객실 등급(pclass)의 분포, **어떤 열에 결측치**가 있는지, 나이·요금의 대략적인 범위 등 눈에 띄는 점을 적으면 됩니다.
- 정답은 하나가 아닙니다. 출력에서 실제로 확인되는 사실이면 됩니다.

> 이 문제는 자가채점(assert)이 없습니다. 아래 서술 셀에 직접 문장을 적고, 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 관찰을 서술하세요)*

## 2. 정규성 검정 (pg.normality — Shapiro-Wilk)
**배경**: 어떤 검정을 쓸지 정하려면 데이터가 **정규분포를 따르는지** 먼저 확인합니다. Pingouin 의 `pg.normality` 는 (기본) Shapiro-Wilk 검정으로 정규성을 봅니다 — 귀무가설은 '데이터가 정규분포를 따른다' 입니다. `age` 가 정규분포를 따른다고 볼 수 있는지 검정해 봅시다.

**요구사항**:
- `data/titanic.csv` 를 읽어 변수 `df` 에 담으세요.
- `age` 열의 **결측치를 제거하고**(`dropna`) 변수 `age` 에 담으세요.
- `pg.normality(age)` 로 정규성을 검정하세요. 결과 표에서 **W**(검정통계량, `['W'].iloc[0]`)를 `shapiro_stat` 에, **pval**(`['pval'].iloc[0]`)을 `shapiro_p` 에 담으세요. (`pg.normality` 는 내부적으로 Shapiro-Wilk 검정이라 W·p 값이 `scipy.stats.shapiro` 와 같습니다.)
- `shapiro_stat` 는 소수 셋째 자리(`round(값, 3)`), `shapiro_p` 는 소수 넷째 자리(`round(값, 4)`)까지 봤을 때 채점됩니다.
- 그리고 **아래 판단 서술 셀**에, 유의수준 0.05 에서 `age` 가 정규분포를 따른다고 볼 수 있는지 결과를 근거로 한 문장으로 적으세요.

**예시**
```
round(shapiro_stat, 3)  →  0.981
round(shapiro_p, 4)     →  0.0
```
<details><summary>힌트</summary>

```text
접근방법:
- 결측을 없앤 age 를 pg.normality 에 넘겨 결과 표에서 W 와 pval 을 꺼낸다.
- p-value 가 유의수준(0.05)보다 작으면 '정규분포' 라는 귀무가설을 기각한다.

세부구현:
1. 파일을 df 로 불러온다
2. age 열에서 결측치를 제거해(dropna) age 에 담는다
3. pg.normality(age) 결과의 ['W'].iloc[0] 를 shapiro_stat, ['pval'].iloc[0] 를 shapiro_p 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(shapiro_stat - 0.981) < 0.01
assert abs(shapiro_p - 0.0) < 0.01
print("✅ 문제2 통과!")

*(여기에 age 가 정규분포를 따른다고 볼 수 있는지 판단을 서술하세요)*

## 3. 1표본 t-검정 (one-sample t-test)
**배경**: 어떤 사람이 '타이타닉 승객의 평균 나이는 30세다'라고 주장합니다. 1표본 t-검정으로 이 주장(모평균 = 30)을 데이터로 검증해 봅시다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `age` 열의 **결측치를 제거**해 `age` 에 담으세요.
- `pg.ttest(age, 30)` 으로 1표본 t-검정을 실행하세요(두 번째 인자에 **기준값 30** 을 스칼라로 넘기면 1표본). 결과 표에서 **T**(`['T'].iloc[0]`)를 `t_stat`, **p_val**(`['p_val'].iloc[0]`)을 `p_value` 에 담으세요.
- `t_stat` 는 `round(값, 3)`, `p_value` 는 `round(값, 4)` 까지 봤을 때 채점됩니다.

**예시**
```
round(t_stat, 3)   →  -0.553
round(p_value, 4)  →  0.5801
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.ttest 의 두 번째 인자에 기준값 30 을 스칼라로 주면 1표본 t-검정이 된다.
- p-value 가 0.05 보다 크면 '평균이 30 이다'라는 귀무가설을 기각하지 못한다.

세부구현:
1. 파일을 df 로 불러오고 age 결측을 제거한다(dropna)
2. pg.ttest(age, 30) 결과의 ['T'].iloc[0] 를 t_stat, ['p_val'].iloc[0] 를 p_value 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(t_stat - (-0.553)) < 0.01
assert abs(p_value - 0.5801) < 0.01
print("✅ 문제3 통과!")

## 4. 독립 2표본 t-검정 — 등분산을 점검해 검정을 고르기
**배경**: '생존자와 사망자의 평균 나이가 다를까?'를 알아봅니다. 두 집단의 평균을 t-검정으로 비교하기 전에, **등분산 가정**을 먼저 점검하고 그 결과에 따라 **적절한 t-검정을 스스로 골라야** 합니다. 두 집단의 분산이 같다고 볼 수 있으면 등분산을 가정하는 **Student**, 다르면 등분산을 가정하지 않는 **Welch** 를 씁니다(`pg.ttest` 의 `correction` 인자로 선택). 효과의 크기는 **Cohen's d** 로 함께 봅니다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `survived` 와 `age` 두 열을 골라 **함께 결측 제거**한 뒤, 사망 그룹(`survived == 0`)의 나이를 `age_died`, 생존 그룹(`survived == 1`)의 나이를 `age_survived` 에 담으세요.
- 먼저 `pg.homoscedasticity(data=sub, dv='age', group='survived')` 로 **등분산(Levene)** 을 점검하세요(`sub` 는 두 열을 결측 제거한 DataFrame). 결과 표에서 **W**(`['W'].iloc[0]`, 검정통계량)를 `levene_stat`, **pval**(`['pval'].iloc[0]`)을 `levene_p` 에 담으세요.
- `levene_p` 를 유의수준 0.05 와 비교해 **Student(`correction=False`) 인지 Welch(`correction=True`) 인지 스스로 판단**하고, 고른 검정을 `pg.ttest(age_died, age_survived, correction=...)` 로 실행하세요. 결과 표에서 **T**(`['T'].iloc[0]`)를 `t_stat`, **p_val**(`['p_val'].iloc[0]`)을 `p_value`, **cohen_d**(`['cohen_d'].iloc[0]`)를 `cohens_d` 에 담으세요 (`cohen_d` 는 부호 없는 크기 |d| — pingouin 이 자동 계산).
- `levene_stat`·`t_stat`·`cohens_d` 는 소수 셋째, `levene_p`·`p_value` 는 소수 넷째 자리까지 봤을 때 채점됩니다.

**예시** — 등분산 점검 결과와, **적절한 검정을 골랐다면** 나오는 값입니다(어떤 검정인지는 스스로 판단).
```
round(levene_stat, 3)  →  1.195
round(levene_p, 4)     →  0.2746   # 0.05 보다 큼 → 등분산을 기각하지 못함
round(t_stat, 3)       →  2.067
round(p_value, 4)      →  0.0391
round(cohens_d, 3)     →  0.157
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 집단의 분산이 유의하게 다른지 등분산 검정으로 먼저 확인한다.
- 등분산 검정의 p 가 유의수준(0.05)보다 작으면 분산이 다르다고 보아 등분산을 가정하지 않는 검정을,
  그렇지 않으면 등분산을 가정하는 검정을 pg.ttest 의 correction 인자로 고른다.

세부구현:
1. survived, age 두 열을 골라 함께 결측 제거해 sub 에 담고, 생존 여부로 두 그룹을 나눈다
2. pg.homoscedasticity(data=sub, dv='age', group='survived') 로 W·pval 을 꺼낸다
3. pval 을 0.05 와 비교해 correction 을 정하고 pg.ttest 로 검정한다
4. 결과 표의 ['T']·['p_val']·['cohen_d'] 를 각각 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(levene_stat - 1.195) < 0.01
assert abs(levene_p - 0.2746) < 0.01
assert abs(t_stat - 2.067) < 0.01, '등분산 점검 결과에 맞는 correction 을 골랐는지 확인하세요'
assert abs(p_value - 0.0391) < 0.01
assert abs(cohens_d - 0.157) < 0.01
print("✅ 문제4 통과!")

## 5. 대응표본 t-검정 (paired t-test)
**배경**: 학습 프로그램에 참가한 15명이 프로그램 전/후로 시험을 봤습니다. 같은 사람을 전/후로 비교하는 것은 서로 다른 두 집단이 아니라 **한 사람 안에서의 변화**를 보는 것이므로, 독립 2표본이 아니라 **대응표본(paired)** t-검정을 씁니다. **이 문제만** 학습 프로그램 참가자 15명의 전/후 점수 데이터(`study_scores.csv`)를 씁니다.

**요구사항**:
- `data/study_scores.csv` 를 읽어 `df` 에 담고 `display(df.head())` 로 간단히 확인하세요.
- `score_after − score_before` 로 차이값을 만들어 `diff` 에 담으세요.
- `pg.normality(diff)` 로 차이값의 정규성을 확인해 **W**(`['W'].iloc[0]`)를 `shapiro_stat`, **pval**(`['pval'].iloc[0]`)을 `shapiro_p` 에 담으세요. (**대응표본은 차이값의 정규성만 확인하면 되고, 등분산 검정은 필요 없습니다.**)
- `pg.ttest(df['score_after'], df['score_before'], paired=True)` 로 대응표본 t-검정을 실행하세요(`paired=True` 가 대응표본). 결과 표에서 **T**(`['T'].iloc[0]`)를 `t_stat`, **p_val**(`['p_val'].iloc[0]`)을 `p_value` 에 담으세요.
- 효과크기 **Cohen's d** 는 결과 표의 **cohen_d** 열(`['cohen_d'].iloc[0]`)을 그대로 `cohens_d` 에 담으세요 — pingouin 이 자동으로 줍니다(손계산 불필요).
- `t_stat`·`cohens_d` 는 `round(값, 3)`, `p_value` 는 `round(값, 4)` 까지 봤을 때 채점됩니다.

**예시**
```
round(t_stat, 3)    →  2.279
round(p_value, 4)   →  0.0389
round(cohens_d, 3)  →  0.399
```
<details><summary>힌트</summary>

```text
접근방법:
- 대응표본은 각 사람의 전/후 차이를 먼저 만들고, 그 차이값의 정규성만 확인한다(등분산 검정 불필요).
- pg.ttest 에 paired=True 를 주고 두 대응 값을 순서대로(after, before) 넘긴다.

세부구현:
1. study_scores.csv 를 df 로 불러오고 head 로 확인한다
2. score_after 에서 score_before 를 빼 diff 를 만든다
3. pg.normality(diff) 결과의 ['W']·['pval'] 를 shapiro_stat, shapiro_p 에 담는다
4. pg.ttest(df['score_after'], df['score_before'], paired=True) 결과의 ['T']·['p_val']·['cohen_d'] 를 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(p_value - 0.0389) < 0.01
assert abs(t_stat - 2.279) < 0.01
assert abs(cohens_d - 0.399) < 0.01
print("✅ 문제5 통과!")

## 6. Wilcoxon 부호순위 검정 — 대응 t 의 비모수 짝
**배경**: 문제 5 에서 같은 학생의 전/후 점수를 **대응 t-검정**으로 비교했습니다. 그 비모수 짝이 **Wilcoxon 부호순위 검정**(`pg.wilcoxon`)이에요. 값 대신 **차이의 순위**만 쓰기 때문에 이상치와 비정규에 강건합니다. 같은 데이터에 두 검정을 나란히 돌려 **결과가 어떻게 다른지** 직접 봅니다.

**요구사항**:
- `data/study_scores.csv` 를 읽어 `df` 에 담으세요.
- `pg.wilcoxon(df['score_after'], df['score_before'])` 로 결과 표를 만들어 `display` 하세요. **문제 5 의 대응 t 와 똑같이 (after, before) 순서**로 넣습니다 — 그래야 두 검정의 방향이 같아집니다.
- 그 표에서 검정통계량 `W_val` 을 `w_stat`, p-value `p_val` 을 `wilcoxon_p` 에 담으세요.
- `w_stat` 은 정수로, `wilcoxon_p` 는 소수 넷째 자리까지 봤을 때 채점됩니다.

**예시**
```
w_stat              →  26.0
round(wilcoxon_p, 4)  →  0.0554
```
> **눈여겨볼 것**: 문제 5 의 대응 t 는 p ≈ 0.0389 로 유의했는데, 여기 Wilcoxon 은 p ≈ 0.0554 로 0.05 를 넘습니다. **같은 데이터인데 결론이 갈립니다.** 왜 그런지는 아래 서술에서 생각해 보세요.
<details><summary>힌트</summary>

```text
접근방법:
- 대응표본의 비모수 검정에 두 열을 그대로 넘긴다. 반환된 표에서 필요한 두 값을 꺼낸다.

세부구현:
1. read_csv 로 데이터를 df 에 담는다
2. 비모수 대응 검정 함수에 문제 5 와 같은 순서(after, before)로 두 열을 넘겨 결과 표를 받는다
3. 결과 표의 W_val 열 첫 값을 w_stat 에, p_val 열 첫 값을 wilcoxon_p 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(w_stat - 26.0) < 0.5
assert abs(wilcoxon_p - 0.0554) < 0.001
print("✅ 문제6 통과!")

**관찰 (서술)**

*(여기에 1~2문장으로 서술하세요 — 같은 데이터인데 대응 t(p≈0.0389)와 Wilcoxon(p≈0.0554)의 결론이 갈린 이유, 그리고 이 데이터에서는 어느 쪽을 믿어야 할지)*

## 7. 정규성을 점검해 검정 고르기 (모수 vs 비모수)
**배경**: '생존자와 사망자의 요금(`fare`)이 다를까?'를 봅니다. 두 집단을 비교할 때, **정규성이 크게 깨지면** 평균을 쓰는 t-검정 대신 **순위 기반의 비모수 검정**이 더 적절합니다. 먼저 각 집단의 정규성을 점검하고, 그 결과에 따라 **적절한 검정을 스스로 골라** 실행하세요.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `survived` 와 `fare` 두 열을 골라 **함께 결측 제거**한 뒤, 사망 그룹(`survived == 0`)의 요금을 `fare_died`, 생존 그룹(`survived == 1`)의 요금을 `fare_survived` 에 담으세요.
- 먼저 `pg.normality` 로 **각 집단 요금의 정규성**을 점검해 pval 을 `shapiro_p_died`·`shapiro_p_survived` 에 담고 출력하세요.
- 두 정규성 결과를 보고 **정규성이 크게 깨졌는지 스스로 판단**한 뒤, 적절한 검정을 골라 실행하세요(정규성을 만족했다면 독립 2표본 t-검정을, 크게 깨졌다면 순위 기반 비모수 검정을 쓰는 것이 원칙). 결과 표에서 검정통계량을 `u_stat`, p-value 를 `p_value` 에 담으세요.
- 검정통계량은 소수 셋째, p-value 는 소수 넷째 자리까지 봤을 때 채점됩니다.
- (고른 검정의 결과 표를 `display` 해, pingouin 이 함께 주는 효과크기도 눈으로 확인하세요.)

**예시** — 정규성 점검 결과와, **적절한 검정을 골랐다면** 나오는 값입니다(어떤 검정인지는 스스로 판단).
```
shapiro_p_died, shapiro_p_survived  →  둘 다 ≈ 0 (0.05 보다 훨씬 작음 → 정규성 강하게 기각)
round(u_stat, 3)   →  57806.5
round(p_value, 4)  →  0.0
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 집단 각각의 정규성을 먼저 확인한다. 정규성 p 가 유의수준(0.05)보다 훨씬 작으면 정규성이 크게 깨진 것이다.
- 정규성이 크게 깨지면 평균 대신 순위로 두 집단을 비교하는 비모수 검정을 쓴다(pg 에서 두 독립표본용 함수).

세부구현:
1. survived, fare 두 열을 골라 함께 결측 제거하고 생존 여부로 두 그룹의 fare 를 나눈다
2. 각 그룹에 pg.normality 를 돌려 pval 을 shapiro_p_died, shapiro_p_survived 에 담아 출력한다
3. 두 정규성 결과로 검정을 정하고 실행하되, 사망(survived==0) 그룹을 첫 인자·생존(survived==1) 그룹을 둘째 인자로 넣어
   결과 표의 검정통계량과 p-value 를 u_stat, p_value 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(u_stat - 57806.5) < 0.5, '정규성 점검 결과에 맞는 검정을 골랐는지, 사망 그룹을 첫 인자로 넣었는지 확인하세요'
assert abs(p_value - 0.0) < 0.01
print("✅ 문제7 통과!")

## 8. 일원분산분석 (One-way ANOVA)
**배경**: 객실 등급(pclass)이 1·2·3 등급으로 나뉠 때, **세 집단의 평균 나이가 모두 같은지** 한 번에 검정하는 것이 분산분석(ANOVA)입니다. 귀무가설은 '세 등급의 평균 나이가 모두 같다' 입니다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담으세요.
- `pclass` 와 `age` 두 열을 골라 **함께 결측 제거**해 `sub` 에 담으세요.
- `pg.anova(data=sub, dv='age', between='pclass', detailed=True)` 로 일원분산분석 표를 만들어 `aov` 에 담고 `display(aov)` 하세요(긴 형태 — 값 열은 `dv`, 그룹 열은 `between`).
- 결과 표 첫 행에서 **F**(`aov['F'].iloc[0]`)를 `f_stat`, **p_unc**(`aov['p_unc'].iloc[0]`)을 `p_value` 에 담으세요.
- `f_stat` 는 `round(값, 3)`, `p_value` 는 `round(값, 4)` 까지 봤을 때 채점됩니다.
- 그리고 **아래 판단 서술 셀**에, 유의수준 0.05 에서 세 등급의 평균 나이가 같다고 볼 수 있는지 결과를 근거로 한 문장으로 적으세요.

**예시**
```
round(f_stat, 3)   →  57.443
round(p_value, 4)  →  0.0
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.anova 는 긴 형태(dv=값 열, between=그룹 열)의 DataFrame 을 받아 F·p·효과크기를 한 표로 준다.
- 결과 표 첫 행의 F 와 p_unc 를 꺼낸다. p_unc<0.05 면 '모든 평균이 같다'를 기각한다.

세부구현:
1. pclass, age 두 열을 골라 함께 결측 제거해 sub 에 담는다
2. pg.anova(data=sub, dv='age', between='pclass', detailed=True) 로 표 aov 를 만든다
3. aov['F'].iloc[0] 를 f_stat, aov['p_unc'].iloc[0] 를 p_value 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(f_stat - 57.443) < 0.01
assert abs(p_value - 0.0) < 0.01
print("✅ 문제8 통과!")

*(여기에 세 등급의 평균 나이가 같다고 볼 수 있는지 판단을 서술하세요)*

## 9. 사후검정 (Tukey HSD)
**배경**: ANOVA 에서 '평균이 다르다'가 나오면, **어느 등급 쌍**이 서로 다른지는 사후검정으로 확인합니다. Tukey HSD 는 모든 쌍을 비교하면서 다중 비교로 인한 오류를 보정해 줍니다. 객실 등급별 나이에 Tukey HSD 를 적용해 봅시다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `pclass` 와 `age` 두 열을 골라 **함께 결측 제거**해 `sub` 에 담으세요.
- `pg.pairwise_tukey(data=sub, dv='age', between='pclass')` 로 사후검정 표를 만들어 `tukey` 에 담고 `display(tukey)` 하세요.
- 결과 표의 **`p_tukey` 열**이 0.05 미만인 쌍이 유의하게 다른 쌍입니다. **유의하게 차이 나는 쌍의 개수**를 `n_significant` 에 정수로 담으세요(`int((tukey['p_tukey'] < 0.05).sum())`).

**예시**
```
display(tukey)  →  A·B·mean_A·mean_B·diff·se·T·p_tukey·hedges 열, 세 쌍(1-2,1-3,2-3) 비교
n_significant   →  3
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.pairwise_tukey 는 모든 그룹 쌍을 비교한 표(DataFrame)를 돌려준다 — 다중검정 보정과 효과크기(hedges) 포함.
- p_tukey 열이 0.05 미만인 행의 개수를 정수로 센다.

세부구현:
1. pclass, age 두 열을 골라 함께 결측 제거해 sub 에 담는다
2. pg.pairwise_tukey(data=sub, dv='age', between='pclass') 로 표 tukey 를 만든다
3. (tukey['p_tukey'] < 0.05).sum() 을 int 로 감싸 n_significant 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_significant == 3
print("✅ 문제9 통과!")

## 10. 카이제곱 독립성 검정 + Cramér's V
**배경**: '성별과 생존 여부는 서로 관련이 있을까?' 두 **범주형 변수**의 관련성은 카이제곱 독립성 검정으로 봅니다. 귀무가설은 '두 변수가 독립이다(관련 없다)' 입니다. 관련의 세기는 **Cramér's V** 로 함께 확인합니다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담으세요.
- `pg.chi2_independence(data=df, x='sex', y='survived')` 로 카이제곱 독립성 검정을 실행하세요. 이 함수는 **(기대빈도표, 관측빈도표, 통계량표)** 세 개를 순서대로 돌려줍니다: `expected, observed, chi_stats = pg.chi2_independence(...)`. 관측 교차표는 `display(observed)` 로 확인하세요.
- 통계량표 `chi_stats` 에서 **표준 Pearson 검정 행**(`chi_stats[chi_stats['test'] == 'pearson']`)을 골라 `chi2`(→`chi2_stat`), `pval`(→`p_value`), `dof`(→`dof`, 정수), `cramer`(→`cramers_v`) 를 꺼내세요. **Cramér's V 는 `cramer` 열로 pingouin 이 자동 계산**해 줍니다(손계산 불필요).
- `chi2_stat`·`cramers_v` 는 `round(값, 3)`, `p_value` 는 `round(값, 4)` 까지 봤을 때 채점됩니다.

**예시**
```
round(chi2_stat, 3)  →  260.717
round(p_value, 4)    →  0.0
round(cramers_v, 3)  →  0.541
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.chi2_independence 는 데이터프레임과 두 범주형 열 이름(x, y)을 받아 (기대, 관측, 통계량) 세 표를 준다.
- 통계량표에서 test=='pearson' 행이 표준 카이제곱 결과다. chi2·pval·cramer 를 그 행에서 꺼낸다.

세부구현:
1. expected, observed, chi_stats = pg.chi2_independence(data=df, x='sex', y='survived')
2. pearson = chi_stats[chi_stats['test'] == 'pearson'] 로 표준 행을 고른다
3. pearson['chi2']·['pval']·['cramer'] 의 .iloc[0] 를 chi2_stat, p_value, cramers_v 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(chi2_stat - 260.717) < 0.01
assert abs(p_value - 0.0) < 0.01
assert abs(cramers_v - 0.541) < 0.01
assert dof == 1
print("✅ 문제10 통과!")

## 11. 카이제곱 적합도 검정 (goodness-of-fit)
**배경**: '객실 등급이 1·2·3 등급에 고르게 분포할까?'를 봅니다. 한 범주형 변수의 관측 분포가 특정 기대 분포(여기서는 **세 등급이 균등**)와 맞는지 보는 것이 적합도 검정입니다. 귀무가설은 '관측 분포가 기대 분포와 같다' 입니다.

> 🔧 **여기만 scipy**: 적합도 검정은 pingouin 이 지원하지 않아 `scipy.stats.chisquare` 를 씁니다. (두 변수 관련성인 **독립성**은 `pg.chi2_independence`(문제 9), 한 변수 분포의 **적합도**는 scipy 로 갈립니다.)

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담으세요.
- `df['pclass'].value_counts().sort_index()` 로 등급별 관측 빈도를 구해 `observed` 에 담으세요 (인덱스를 정렬해 1·2·3 등급 순으로).
- 세 등급이 **균등하다고 가정**한 기대 빈도를 만드세요: 전체 수 `n = observed.sum()` 을 3 으로 나눈 값을 세 번 반복한 배열 `expected`(예: `np.repeat(n/3, 3)`).
- `stats.chisquare(observed.values, f_exp=expected)` 로 검정통계량을 `chi2_stat`, p-value 를 `p_value` 에 담으세요.
- `chi2_stat` 는 `round(값, 3)`, `p_value` 는 `round(값, 4)` 까지 봤을 때 채점됩니다.

**예시**
```
round(chi2_stat, 3)  →  191.805
round(p_value, 4)    →  0.0
```
<details><summary>힌트</summary>

```text
접근방법:
- 등급별 개수를 세어 관측 빈도를 만들고, '균등'이라는 기대 빈도(전체÷3)를 만든다.
- 관측 합과 기대 합이 같아야 한다(둘 다 전체 표본 수).

세부구현:
1. value_counts().sort_index() 로 등급별 관측 빈도 observed 를 만든다
2. n = observed.sum() 을 3 으로 나눈 값을 세 번 반복해 expected 를 만든다
3. chisquare 에 observed.values 와 f_exp=expected 를 넘겨 chi2_stat, p_value 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(chi2_stat - 191.805) < 0.01
assert abs(p_value - 0.0) < 0.01
print("✅ 문제11 통과!")

## 12. 정규성 시각화 — Q-Q Plot (pg.qqplot)
**배경**: 정규성은 `pg.normality` 같은 검정뿐 아니라 **Q-Q Plot** 으로 눈으로도 확인합니다. 점들이 빨간 기준선(정규분포)에 가까이 놓이면 정규분포에 가깝고, 양 끝이 선에서 벗어나면 꼬리가 두껍거나 치우친 것입니다. Pingouin 의 `pg.qqplot` 으로 `age` 의 Q-Q Plot 을 그려 봅시다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `age` 열의 **결측치를 제거**해 `age` 에 담으세요.
- `fig, ax = plt.subplots(figsize=(6, 5))` 로 새 그림·축을 연 뒤, `pg.qqplot(age, dist='norm', ax=ax)` 로 Q-Q Plot 을 그리세요.
- `ax.set_title(...)` 로 제목을 달고 `plt.show()` 를 호출하세요.

**예시**: 이론 분위수(정규분포)와 실제 데이터 분위수를 견주는 Q-Q Plot 으로, pingouin 은 빨간 기준선과 R²(적합도)까지 함께 그려 줍니다. 아래 완성 그래프와 같은 모양이면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 결측을 없앤 age 를 pg.qqplot 에 넘기되 ax= 에 미리 만든 축을 준다.
- 그리기 전에 plt.subplots 로 새 그림·축을 연다.

세부구현:
1. 파일을 df 로 불러오고 age 결측을 제거한다(dropna)
2. fig, ax = plt.subplots 로 새 그림·축을 연다
3. pg.qqplot(age, dist='norm', ax=ax) 를 그린 뒤 ax.set_title 로 제목을 달고 plt.show 로 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv1_q12.png" width="520">

In [ ]:
# 여기에 코드를 작성하세요

## 13. 미니 통합 — 가정 점검부터 효과크기까지
**배경**: 지금까지 배운 흐름을 **한 문제에 묶어** 봅니다. '1등급과 3등급 승객의 평균 나이가 다를까?'를 **가정 점검 → 적절한 검정 선택 → 실행 → 효과크기 해석** 의 순서로 스스로 완성하세요. 어떤 검정을 쓸지는 **점검 결과를 보고 직접 판단**합니다.

**요구사항**:
- `data/titanic.csv` 를 읽어 `df` 에 담고, `pclass` 가 **1 또는 3** 인 행만 남긴 뒤 `pclass`·`age` 두 열의 결측을 제거해 `sub` 에 담으세요. 1등급 나이를 `age_first`(`pclass == 1`), 3등급 나이를 `age_third`(`pclass == 3`) 에 담으세요.
- **① 정규성 점검**: `pg.normality` 로 두 집단의 정규성 pval 을 `norm_p_first`·`norm_p_third` 에 담아 출력하세요. (표본이 각각 100개가 넘어 크므로, 정규성이 다소 깨져도 **중심극한정리** 덕분에 평균을 비교하는 t-검정이 강건합니다 — 그래서 여기서는 **t-검정 계열**로 갑니다.)
- **② 등분산 점검**: `pg.homoscedasticity(data=sub, dv='age', group='pclass')` 로 **W**(`['W'].iloc[0]`)를 `levene_stat`, **pval**(`['pval'].iloc[0]`)을 `levene_p` 에 담으세요.
- **③ 검정 선택·실행**: `levene_p` 를 0.05 와 비교해 **Student(`correction=False`)/Welch(`correction=True`)** 중 적절한 것을 **스스로 골라** `pg.ttest(age_first, age_third, correction=...)` 를 실행하세요. 결과 표에서 **T**·**p_val**·**cohen_d** 를 `t_stat`·`p_value`·`cohens_d` 에 담으세요.
- **④ 효과크기 해석**: 아래 서술 셀에 `cohens_d` 의 크기를 기준(0.2/0.5/0.8 = 작은/중간/큰)과 견줘 한 문장으로 해석하세요.
- 검정통계량·효과크기는 소수 셋째, p값은 소수 넷째 자리까지 봤을 때 채점됩니다.

**예시** — 점검 결과와, **적절한 검정을 골랐다면** 나오는 값입니다(어떤 검정인지는 스스로 판단).
```
norm_p_first  →  ≈ 0.364 (0.05 보다 큼)     norm_p_third  →  ≈ 0 (0.05 보다 작음)
round(levene_stat, 3)  →  11.78
round(levene_p, 4)     →  0.0006   # 0.05 보다 작음 → 두 집단의 분산이 다름
round(t_stat, 3)       →  10.293
round(p_value, 4)      →  0.0
round(cohens_d, 3)     →  0.982
```
<details><summary>힌트</summary>

```text
접근방법:
- pclass 가 1 또는 3 인 행만 남기고 두 집단의 나이를 나눈다.
- 정규성은 각 집단에, 등분산은 긴 형태(dv=age, group=pclass)로 점검한다.
- 표본이 크면 정규성이 다소 깨져도 t-검정이 강건하므로 t-검정 계열을 쓰되,
  등분산 검정의 p 가 유의수준보다 작으면 등분산을 가정하지 않는 correction 을, 그렇지 않으면 가정하는 correction 을 고른다.

세부구현:
1. pclass 가 [1, 3] 인 행만 남기고 pclass·age 결측을 제거해 sub 에 담는다
2. 1등급·3등급 나이를 age_first, age_third 에 담는다
3. 각 집단에 pg.normality 를 돌려 pval 을 norm_p_first, norm_p_third 에 담아 출력한다
4. pg.homoscedasticity 로 W·pval 을 levene_stat, levene_p 에 담는다
5. levene_p 를 0.05 와 비교해 correction 을 정하고 pg.ttest 로 검정해 T·p_val·cohen_d 를 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(levene_stat - 11.78) < 0.01
assert abs(levene_p - 0.0006) < 0.01
assert abs(t_stat - 10.293) < 0.01, '등분산 점검 결과에 맞는 correction 을 골랐는지 확인하세요'
assert abs(p_value - 0.0) < 0.01
assert abs(cohens_d - 0.982) < 0.01
print("✅ 문제13 통과!")

*(여기에 cohens_d 의 크기를 기준(0.2/0.5/0.8)과 견줘 효과크기를 해석하세요)*

## 14. 검정 결과 해석하기 (서술형)
**배경**: 검정을 돌리는 것만큼 중요한 것이 **결과를 올바로 해석**하는 일입니다. 앞에서 얻은 두 결과를 비교해 봅시다.
- 문제 3 (1표본 t-검정): p-value ≈ **0.5801**
- 문제 4 (독립 2표본 t-검정): p-value ≈ **0.0391**

**요구사항**: 아래 서술 셀에 다음을 포함해 3~4문장으로 적으세요.
- 각 결과가 유의수준 0.05 에서 귀무가설을 **기각하는지/기각하지 못하는지**.
- **p-value** 가 무엇을 의미하는지(작을수록 무엇을 뜻하는지) 자신의 말로.
- '통계적으로 유의하다'와 '실질적으로 의미가 크다'가 왜 다른지 (문제 4 의 Cohen's d ≈ 0.16 을 근거로).

> 이 문제는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 두 검정 결과의 해석을 서술하세요)*

## 15. 어떤 상황에 어떤 검정? (서술형)
**배경**: 이 단원에서 배운 검정들은 각각 쓰임이 다릅니다. 다음 네 상황에 **어떤 검정**을 쓸지, 그리고 **왜** 그 검정인지 한 줄씩 적어 보세요.

**요구사항**: 아래 서술 셀에 (가)~(라) 각각에 대해 '적절한 검정 + 이유'를 적으세요.
- (가) 두 집단(예: 생존/사망)의 평균을 비교하는데, 데이터가 **정규분포를 따르고** 분산이 다를 수 있다.
- (나) 두 집단을 비교하는데, 데이터가 **정규성을 심하게 위반**한다(치우침·이상치).
- (다) **세 집단 이상**의 평균을 한 번에 비교한다.
- (라) 두 **범주형 변수**(예: 성별과 생존 여부)의 관련성을 본다.

> 이 문제는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 (가)~(라)의 적절한 검정과 이유를 서술하세요)*